# Day 3 — Functions & Scope### Python for Data Science · Module 1 · Topic 1.3**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University---**Session length:** 2 hours**Format:** 90 min concepts + live coding · 30 min practice| # | What we cover | Time ||---|---|---|| 1 | Defining & calling: `def`, parameters, `return` vs `print` | 20 min || 2 | Arguments: positional, keyword, defaults, `*args`, `**kwargs` | 30 min || 3 | Scope: local vs global, the LEGB rule | 25 min || 4 | Writing them well: docstrings, type hints, `lambda` | 10 min || 5 | Mini build: refactoring Day 1's result calculator | 5 min || 6 | **Practice notebook (separate file)** | 30 min |> **The habit to build today:** write the function signature and docstring **first**, before any logic.> Deciding what goes in and what comes out is most of the design work.

---## 0. Recap of Day 2Answer in your head before running.

In [ ]:
print("range(1,5) ->", list(range(1, 5)))     # 1,2,3,4 - stop excludedout = []for n in [1, 2, 3, 4, 5]:    if n == 3: continue      # skips ONE item    if n == 5: break         # stops the loop    out.append(n)print("continue/break ->", out)               # [1, 2, 4]nums = [1, 2, 3]for n in nums:    n = n * 2                # n is a copyprint("loop variable ->", nums)               # unchanged

A function body is just another indented block — yesterday's indentation work carries straight over.

---# 1. Defining & Calling## 1.1 Why functions? Because copy-paste does not scale

In [ ]:
# WITHOUT a function - the formula is written out every timetotal = 88 + 71 + 64pct   = total / 300 * 100print(f"Ravi: {pct:.1f}%")total = 91 + 84 + 79pct   = total / 300 * 100print(f"Sara: {pct:.1f}%")# Now imagine the exam becomes out of 400. How many places do you edit?

In [ ]:
# WITH a function - the formula lives in exactly one placedef percentage(a, b, c):    total = a + b + c    return total / 300 * 100print(f"Ravi: {percentage(88, 71, 64):.1f}%")print(f"Sara: {percentage(91, 84, 79):.1f}%")# One edit fixes every caller.

**Four wins:** don't repeat yourself · one place to fix a bug · readable names · testable in isolation.

## 1.2 Anatomy of a function```def   greet   (name)   : │      │        │     └─ opens the block; body is indented │      │        └─────── parameter list │      └──────────────── the name (snake_case) └─────────────────────── the keyword that starts a definition```

In [ ]:
def greet(name):    message = f"Hello, {name}!"    return messageresult = greet("Ravi")print(result)

### ⚠️ Defining is not runningA very common first-week confusion: you write a perfect function, run the cell, see no output,and assume it is broken. The definition only teaches Python the recipe. **Nothing cooks until you call it.**

In [ ]:
def say_hello():    print("Hello!")# Running this cell defines the function. Notice: NO output yet.print("Cell finished - but nothing was printed by the function.")

In [ ]:
say_hello()      # NOW the body runs. Parentheses are what make it a call.print("---")print(say_hello)     # without () you get the function object itself, not a call

## 1.3 `return` vs `print` — the distinction that matters most today

In [ ]:
# VERSION A: print inside the functiondef add_print(a, b):    print(a + b)result = add_print(2, 3)     # displays 5print("result is:", result)  # None!  - the value was never handed back

In [ ]:
# Trying to USE that result failsdef add_print(a, b):    print(a + b)result = add_print(2, 3)try:    print(result * 2)except TypeError as e:    print("TypeError:", e)

In [ ]:
# VERSION B: return the valuedef add(a, b):    return a + bresult = add(2, 3)           # nothing displayedprint("result is:", result)  # 5print("doubled:", result * 2)          # 10 - it is a real numberprint("composable:", add(1, 2) + add(3, 4))    # 10

> ### The rule> - `print` is for a **human to read**.> - `return` is for the **rest of your program to use**.>> Almost every function you write should `return`.>> A function with no `return` statement returns `None` automatically. That `None` is where> the mysterious `TypeError: unsupported operand type(s) for *: 'NoneType'` comes from —> you tried to do arithmetic on nothing.

In [ ]:
def no_return():    x = 42          # computed, but never handed backprint(no_return())          # Noneprint(no_return() is None)  # True

## 1.4 Three things `return` does that surprise people

In [ ]:
# 1. return exits IMMEDIATELY - nothing after it runsdef check(n):    return "positive"    print("this line never runs")     # unreachableprint(check(5))

In [ ]:
# 2. You can return several values at oncedef stats(nums):    return min(nums), max(nums), sum(nums)lo, hi, tot = stats([3, 9, 4, 7])print(lo, hi, tot)# Python packs them into a tuple, then unpacks - Day 1's swap trick againpacked = stats([3, 9, 4, 7])print(packed, type(packed))

In [ ]:
# 3. Early return replaces deep nesting# Nested version - main logic buried three levels deepdef grade_nested(m):    if 0 <= m <= 100:        if m >= 75:            return "A"        else:            if m >= 40:                return "C"            else:                return "F"    else:        return "invalid"# Early-return version - handle bad cases first, then assume clean inputdef grade(m):    if not 0 <= m <= 100:        return "invalid"        # leave immediately    if m >= 75:        return "A"    if m >= 40:        return "C"    return "F"for m in [90, 55, 20, 150]:    print(m, "->", grade(m))

---# 2. Arguments**Parameters** are the names in the definition. **Arguments** are the values you pass when calling.```pythondef introduce(name, age, city):     # name, age, city are PARAMETERSintroduce("Ravi", 23, "Pune")       # those three values are ARGUMENTS```

## 2.1 Positional vs keyword arguments

In [ ]:
def introduce(name, age, city):    return f"{name}, {age}, from {city}"# POSITIONAL - order is everythingprint(introduce("Ravi", 23, "Pune"))     # correct# Swap two of them and you get NO ERROR - just nonsenseprint(introduce(23, "Ravi", "Pune"))     # name=23, age="Ravi"

In [ ]:
# KEYWORD - order stops mattering, and the call documents itselfprint(introduce(name="Ravi", age=23, city="Pune"))print(introduce(city="Pune", name="Ravi", age=23))    # identical# You can mix - but every positional must come BEFORE every keywordprint(introduce("Ravi", city="Pune", age=23))# introduce(name="Ravi", 23, "Pune")#   -> SyntaxError: positional argument follows keyword argument

**Practical advice:** on any call with more than two arguments, use keywords.It costs a few characters and removes an entire category of silent bugs.

## 2.2 Default arguments

In [ ]:
def greet(name, greeting="Hello"):    return f"{greeting}, {name}!"print(greet("Ravi"))                  # uses the defaultprint(greet("Ravi", "Welcome"))       # overrides it positionallyprint(greet("Ravi", greeting="Hi"))   # overrides it by keyword

Defaults are why `train_test_split(X, y)` works with no other arguments —every remaining parameter has a sensible default. You will meet this constantly in Module 3.### ⚠️ Defaults must come AFTER non-default parameters

In [ ]:
# def greet(greeting="Hello", name):#   -> SyntaxError: non-default argument follows default argument# Python fills positional arguments left to right, so it cannot work out what a# bare value means once an optional parameter sits ahead of a required one.# Required parameters first, optional ones after. Always.def greet(name, greeting="Hello"):     # correct order    return f"{greeting}, {name}!"print(greet("Sara"))

## 2.3 ⚠️ The famous mutable default trapThis is the single most surprising thing in today's session. Run it and watch.

In [ ]:
def add_item(item, basket=[]):        # looks completely reasonable    basket.append(item)    return basketprint(add_item("apple"))    # ['apple']print(add_item("bread"))    # ['apple', 'bread']   ?!print(add_item("milk"))     # ['apple', 'bread', 'milk']# Each call was supposed to start with an empty basket.

**Why:** the default value is created **once**, when the function is *defined* — not on each call.So every call that omits the argument shares the exact same list object.This is **aliasing**: two names pointing at one object, exactly like `c = a` on Day 1.

In [ ]:
# Proof: the default object is the same one every timedef show_id(basket=[]):    return id(basket)print(show_id())print(show_id())      # identical id - it is literally the same list

In [ ]:
# THE FIX: default to None, build a fresh object insidedef add_item(item, basket=None):    if basket is None:        basket = []           # a new list on every call    basket.append(item)    return basketprint(add_item("apple"))    # ['apple']print(add_item("bread"))    # ['bread']   correctprint(add_item("milk"))     # ['milk']

> **Rule:** never use a list, dict or set as a default value. Use `None` and build it inside.

## 2.4 `*args` and `**kwargs`

In [ ]:
# *args collects any extra POSITIONAL arguments into a TUPLEdef total(*args):    print("  received:", args, type(args).__name__)    return sum(args)print(total(1, 2))print(total(1, 2, 3, 4))print(total())            # empty tuple -> 0

In [ ]:
# **kwargs collects any extra KEYWORD arguments into a DICTdef profile(**kwargs):    print("  received:", kwargs, type(kwargs).__name__)    for key, value in kwargs.items():        print(f"    {key}: {value}")profile(name="Ravi", age=23, city="Pune")

> The `*` and `**` are the actual syntax. `args` and `kwargs` are just conventional names —> `*nums` and `**options` work exactly the same. Students often think the words are magic keywords.

In [ ]:
# The order is fixed: positional -> *args -> keyword-only -> **kwargsdef describe(title, *items, sep=", ", **extras):    line = f"{title}: " + sep.join(str(i) for i in items)    if extras:        line += f"  {extras}"    return lineprint(describe("Marks", 88, 71, 64))print(describe("Marks", 88, 71, 64, sep=" | "))print(describe("Marks", 88, 71, 64, student="Ravi", term=2))

You will meet `*args` / `**kwargs` constantly in library code — it is how a function like`plt.plot()` accepts almost anything. Writing them yourself is rarer; reading them is a daily skill.

---# 3. Scope## 3.1 Local and global

In [ ]:
course = "Data Science"      # GLOBAL - defined at the top leveldef show():    room = "B-204"           # LOCAL - exists only inside this function    print("course:", course) # a function CAN read a global    print("room:  ", room)show()

In [ ]:
# But the global level cannot see inside the functiontry:    print(room)except NameError as e:    print("NameError:", e)

### Why this is a feature, not a restrictionBecause locals are sealed off, you can use `total` as a variable name inside fifty differentfunctions and they will never collide. Each call gets its own private workspace, created onentry and thrown away on return.Without scope, every function in every library you import could clobber your variables.

In [ ]:
def f():    total = "I am f's total"    return totaldef g():    total = "I am g's total"    return totaltotal = "I am the global total"print(f())print(g())print(total)      # all three are completely independent

## 3.2 The LEGB rulePython looks a name up in this order and **stops at the first match**:| | Scope | Where ||---|---|---|| **L** | Local | Inside the current function || **E** | Enclosing | Inside any outer function wrapping it || **G** | Global | Top level of the file or notebook || **B** | Built-in | Python's own names: `len`, `print`, `sum`, `range` |

In [ ]:
x = "global"def outer():    x = "enclosing"    def inner():        x = "local"        print("inner sees:", x)    inner()    print("outer sees:", x)outer()print("top level sees:", x)

> ### This explains Day 1's shadowing warning> Naming a variable `list` or `sum` breaks things because your **G**lobal name is found> before Python's **B**uilt-in — so the original function is no longer reachable.>> Two separate warnings, one underlying idea.

In [ ]:
sum = 100                  # a global named sum now shadows the built-intry:    print(sum([1, 2, 3]))except TypeError as e:    print("TypeError:", e)del sum                    # remove it and the built-in is reachable againprint("restored:", sum([1, 2, 3]))

## 3.3 Assigning creates a local — always

In [ ]:
count = 0def increment():    count = count + 1       # looks like it should work    return counttry:    increment()except UnboundLocalError as e:    print("UnboundLocalError:", e)# WHY: assigning to `count` ANYWHERE in the body makes it local for the WHOLE# function. So the right-hand side tries to read a local that does not exist yet.

In [ ]:
# GOOD: take it in, hand it backdef increment(count):    return count + 1count = 0count = increment(count)count = increment(count)print("count is now:", count)      # explicit - you can see where it changed

In [ ]:
# The `global` keyword also works, but hides where the change came fromcount = 0def increment_global():    global count    count += 1increment_global()increment_global()print("count is now:", count)      # 2# Avoid this in your own code. Passing in and returning is clearer and easier to test.

### ⚠️ But mutable objects CAN be changed from inside — no `global` needed

In [ ]:
def add_one(items):    items.append(1)        # MUTATES the caller's listnums = [10, 20]add_one(nums)print(nums)                # [10, 20, 1] - changed!# Rebinding a name (=) is local.# Mutating an object (.append, items[i] = x) reaches through to the original,# because both names point at the SAME object.

In [ ]:
# If you do not want that, pass a copydef add_one_safe(items):    items = items[:]       # a shallow copy    items.append(1)    return itemsnums = [10, 20]result = add_one_safe(nums)print("original:", nums)      # unchangedprint("returned:", result)

> ### One idea, three appearances> Day 1's `c = a` aliasing · today's mutable default trap · this slide's mutable argument.> They are all the same thing: **two names, one object**. Tomorrow's mutability topic makes it four.

---# 4. Writing functions well## 4.1 Docstrings and type hints

In [ ]:
def percentage(marks: list, out_of: int = 100) -> float:    """Return the percentage score for a list of marks.    Args:        marks:  list of individual subject marks        out_of: maximum marks per subject (default 100)    Returns:        The percentage as a float, e.g. 74.33    """    return sum(marks) / (len(marks) * out_of) * 100print(percentage([88, 71, 64]))print(f"{percentage([88, 71, 64]):.2f}%")

In [ ]:
# Your docstring is what help() printshelp(percentage)

**In Colab:** type `percentage(` and press `Shift + Tab` to see your own docstring as a tooltip.That is the payoff for writing them.**Type hints** (`marks: list`, `-> float`) are documentation, not enforcement — Python does notcheck them at runtime. Editors use them to catch mistakes before you run anything.> ### One function, one job> If your docstring needs the word "and" twice to describe what the function does,> it is really two functions. A function that loads a file, cleans it, trains a model and> prints a chart is impossible to test and impossible to reuse.

## 4.2 `lambda` — a one-line function with no name

In [ ]:
# These two are equivalentdef square(n):    return n * nsquare_lambda = lambda n: n * nprint(square(5), square_lambda(5))# Syntax:  lambda <params>: <single expression># The result is returned automatically - there is no `return` keyword.

In [ ]:
# Where you will ACTUALLY meet it: as an argument to another functionstudents = [("Ravi", 88), ("Sara", 91), ("Amit", 76)]students.sort(key=lambda s: s[1])       # sort by the score (index 1)print(students)students.sort(key=lambda s: s[0])       # sort by the nameprint(students)

> **Use it as an argument, not as a replacement for `def`.**>> ```python> square = lambda n: n * n      # pointless - just use def> sort(key=lambda s: s[1])      # exactly right> ```>> A lambda holds one expression and cannot contain statements — no loops, no `if` blocks,> no multiple lines. If you find yourself fighting that limit, you wanted a real function.>> You will use it constantly with pandas `.apply()` in Topic 1.15.

---# 5. Putting it together — refactoring Day 1's result calculatorCompare this with the single 25-line block from Day 1. Same output — but now each piececan be tested, reused and fixed on its own.

In [ ]:
def percentage(marks, out_of=100):    """Return the percentage for a list of marks."""    return sum(marks) / (len(marks) * out_of) * 100def grade(pct):    """Map a percentage to a letter grade."""    if not 0 <= pct <= 100:        return "invalid"          # early return    if pct >= 75:        return "A"    elif pct >= 60:        return "B"    elif pct >= 40:        return "C"    return "F"def report(name, marks):    """Print one student's result line."""    pct = percentage(marks)    print(f"{name:8} {pct:6.2f}%   {grade(pct)}")students = {    "Ravi": [88, 71, 64],    "Sara": [91, 84, 79],    "Amit": [45, 38, 52],}print(f"{'NAME':8} {'PCT':>7}   GRADE")print("-" * 26)for name, marks in students.items():    report(name, marks)

Notice what each piece contributes:- **Three small functions**, each doing exactly one job- **Docstrings** — one line each is enough- **Default argument** `out_of=100` keeps the common call short- **Early return** in `grade()` handles invalid input immediately- **Composition** — `report()` calls the other two- **Day 2's loop** with `.items()` drives the whole thing

---# 6. Recap — the twelve things to remember1. `def` defines; parentheses call. Both are needed.2. `return` hands a value back; `print` only displays it.3. No `return` statement means the function returns `None`.4. `return` exits the function immediately — code after it never runs.5. You can return several values at once; they arrive as a tuple.6. Keyword arguments make a call self-documenting and order-proof.7. Defaults go last, and must never be a list, dict or set.8. `*args` collects a tuple; `**kwargs` collects a dict.9. Locals are private — each call gets a fresh workspace.10. LEGB: Local, Enclosing, Global, Built-in.11. Assigning makes a local; mutating reaches through to the original object.12. One function, one job. Docstring first, body second.---### 📝 Now open **`Day3_Practice_Questions.ipynb`** for the 30-minute practice session.### Homework- Rewrite Day 2's guessing game using three functions.- Write `is_valid_email(text)` returning `True`/`False` (must contain `@` and end in `.com`).- Add docstrings to everything you wrote today.### Next class — Topic 1.4: Lists, Tuples & SetsCreating, indexing, slicing, mutability, and the methods you will use on every dataset.---*Slides & notebooks by Srinivasa Sai Chava · Boston University*